# Task 3: Research Results, Data Analysis, and Scientific Visualization
**Project:** Fast Dopamine & Academic Performance — GPA Prediction Benchmark  
**Repository:** `https://github.com/EraMiir/fast-dopamine-academic-performance`

This notebook executes the complete reproducible analysis pipeline by importing the modular functions from `src/`:
1. **Data Ingestion & Warm-up Filtering (`src/process_data.py`):** Loading raw benchmark logs (`data/raw/test_run.jsonl`) generated from the real student survey dataset (`data/cleaned_survey.csv`, $N=256$, 24 features) and discarding cold-start `warmup` runs.
2. **Statistical Aggregation (`src/process_data.py`):** Computing Mean, SD, Median, Percentiles ($p_{25}, p_{75}, p_{95}, p_{99}$), 1,000-iteration Bootstrapped 95% Confidence Intervals (CI), and Guardrail Error Rates.
3. **Scientific Visualization & Export (`src/visualize.py`):** Generating colorblind-safe plots with explicit physical units, log scales, and 95% CI bands exported to `.pdf` and 300 DPI `.png` in `reports/figures/`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Fix random seed for 100% reproducibility
np.random.seed(42)

# Add src/ to Python path and import modular pipeline functions
root_dir = Path('..').resolve() if Path('../src').exists() else Path('.').resolve()
sys.path.insert(0, str(root_dir / 'src'))

from process_data import process_benchmark_logs
from visualize import generate_all_plots

# 1. Data Ingestion & Warm-up Filtering
raw_jsonl_path = str(root_dir / 'data/raw/test_run.jsonl')
processed_csv_path = str(root_dir / 'data/processed/metrics_summary.csv')

raw_df, eval_df, summary_df = process_benchmark_logs(
    raw_jsonl_path=raw_jsonl_path,
    output_csv_path=processed_csv_path,
    n_boot=1000,
    seed=42
)

print(f'Total raw records loaded: {len(raw_df)}')
print(f'Warm-up records discarded: {len(raw_df) - len(eval_df)}')
print(f'Clean evaluation records retained: {len(eval_df)}')

Total raw records loaded: 276
Warm-up records discarded: 36
Clean evaluation records retained: 240


In [2]:
# 2. Inspect Statistical Aggregation Results (including p95, p99, MAE, and 95% Bootstrap CI)
cols_to_show = [
    'model', 'RMSE_mean', 'RMSE_sd', 'RMSE_median', 'RMSE_p95', 'RMSE_p99',
    'MAE_mean', 'latency_median_ms', 'latency_p95_ms', 'latency_p99_ms', 'error_rate'
]
print('Aggregated Metrics Summary (Sample Size N=256):')
print(summary_df[summary_df['sample_size'] == 256][cols_to_show].to_string(index=False))

Aggregated Metrics Summary (Sample Size N=256):
           model  RMSE_mean  RMSE_sd  RMSE_median  RMSE_p95  RMSE_p99  MAE_mean  latency_median_ms  latency_p95_ms  latency_p99_ms  error_rate
  Baseline_Ridge     0.3338   0.0245       0.3335    0.3731    0.3766    0.2677              1.069           1.491           1.601         0.0
GradientBoosting     0.2918   0.0278       0.2919    0.3402    0.3425    0.2192             80.636          94.740          96.930         0.0
    RandomForest     0.2866   0.0220       0.2926    0.3153    0.3160    0.2221            173.454         186.631         187.356         0.0


In [3]:
# 3. Scientific Visualization & Export (.pdf & 300 DPI .png)
figures_dir = str(root_dir / 'reports/figures')
generate_all_plots(eval_df, summary_df, out_dir=figures_dir)
print(f'Generated fig1_latency_vs_load and fig2_distribution_boxplot (.pdf & .png) in ../reports/figures')

Generated fig1_latency_vs_load and fig2_distribution_boxplot (.pdf & .png) in ../reports/figures
